# Camada Gold Financeira — Terra Signal

Nesta etapa, serão preparados dados financeiros a partir da camada Silver, com o objetivo de analisar a receita associada aos clientes, os impactos dos cancelamentos e os segmentos que merecem maior atenção.

O objetivo é organizar informações financeiras para a construção de indicadores e análises que auxiliem na identificação de possíveis perdas de receita e oportunidades de retenção de clientes.

### Considerações sobre os dados

A base utilizada não apresenta as datas dos cancelamentos. Portanto, os valores mensais associados aos clientes que cancelaram serão tratados como estimativas de receita mensal potencialmente perdida, e não como perdas financeiras efetivamente registradas em determinado mês.

Os registros com valores acumulados ausentes serão considerados apenas nas análises em que existam informações suficientes, evitando a criação de valores sem fundamento.

In [1]:
import pandas as pd
from pathlib import Path

pasta_silver = Path("dados/silver")

df_financeiro = pd.read_csv(pasta_silver / "history_silver.csv")

df_financeiro.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome,TotalCharges_Estimado,tenure_numerico
0,4578-PHJYZ,Male,0,Yes,Yes,52,Yes,No,DSL,No,...,One year,Yes,Electronic check,68.75,3482.85,No,I have been a customer with this internet prov...,6532,3482.85,52.0
1,6289-CPNLD,Male,0,Yes,Yes,33,Yes,No,DSL,No,...,One year,Yes,Mailed check,73.90,2405.05,Yes,I've been a customer with this company for ove...,7634,2405.05,33.0
2,2682-KEVRP,Female,1,No,No,22,Yes,No,No,No internet service,...,One year,Yes,Mailed check,20.05,417.00,No,I have been a customer with this company for 2...,3628,417.00,22.0
3,5697-GOMBF,Female,1,Yes,Yes,28,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,35.90,973.65,No,I have been a customer with this internet prov...,7851,973.65,28.0
4,9717-QEBGU,Male,0,No,No,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.30,144.00,No,I have been using the fiber optic internet ser...,1691,144.00,2.0


- import pandas as pd: importa a biblioteca Pandas.
- from pathlib import Path: permite trabalhar com caminhos de arquivos.
- pasta_silver: indica onde está nosso arquivo tratado.
- pd.read_csv(): carrega os dados da Silver para um DataFrame chamado df_financeiro.
- df_financeiro.head(): apresenta os cinco primeiros registros para conferência.

Por que df_financeiro?

Porque vamos utilizar esse DataFrame como ponto de partida para preparar nossas tabelas financeiras, sem modificar o arquivo original da Silver.

### Seleção das variáveis financeiras

Nesta etapa, serão selecionadas as colunas da camada Silver necessárias para as análises financeiras.

Serão utilizadas informações sobre identificação dos clientes, tipos de contrato, serviços de internet, cancelamentos, valores mensais, valores acumulados e tempo de permanência.

A seleção permite trabalhar apenas com os dados relevantes para os indicadores financeiros, mantendo a base original da camada Silver preservada.

In [2]:
df_financeiro = df_financeiro[[
    "customerID",
    "Contract",
    "InternetService",
    "Churn",
    "MonthlyCharges",
    "TotalCharges_Estimado",
    "tenure_numerico"
]].copy()

df_financeiro.head()

,customerID,Contract,InternetService,Churn,MonthlyCharges,TotalCharges_Estimado,tenure_numerico
0,4578-PHJYZ,One year,DSL,No,68.75,3482.85,52.0
1,6289-CPNLD,One year,DSL,Yes,73.90,2405.05,33.0
2,2682-KEVRP,One year,No,No,20.05,417.00,22.0
3,5697-GOMBF,Month-to-month,DSL,No,35.90,973.65,28.0
4,9717-QEBGU,Month-to-month,Fiber optic,No,70.30,144.00,2.0


### Análise da receita mensal por situação do cliente

Nesta etapa, será analisada a receita mensal associada aos clientes que permaneceram e aos que cancelaram os serviços.

A análise utiliza os valores da coluna `MonthlyCharges`, agrupados conforme a situação de cancelamento (`Churn`).

O objetivo é identificar o valor mensal associado a cada grupo e estimar a receita mensal potencialmente perdida com os cancelamentos.

Como a base não apresenta as datas dos cancelamentos, os valores dos clientes que cancelaram serão considerados estimativas de receita mensal potencialmente perdida, e não perdas financeiras efetivamente registradas em um determinado mês.

In [3]:
tabela_receita = df_financeiro.groupby("Churn")["MonthlyCharges"].sum().round(2)

tabela_receita

Churn
No     251767.15
Yes    112242.90
Name: MonthlyCharges, dtype: float64

### Resultado da análise da receita mensal

Os clientes que permaneceram apresentaram um valor mensal acumulado de R$ 251.767,15, enquanto os clientes que cancelaram representavam R$ 112.242,90 em mensalidades.

Somando os dois grupos, obtemos um valor mensal total de R$ 364.010,05.

Os clientes que cancelaram representavam aproximadamente 30,84% desse valor total.

Como não temos as datas dos cancelamentos, os R$ 112.242,90 representam uma estimativa de receita mensal potencialmente perdida, e não uma perda efetivamente registrada em determinado mês.

### Indicadores de receita mensal

Nesta etapa, serão calculados indicadores financeiros a partir dos valores mensais associados aos clientes que permaneceram e aos que cancelaram os serviços.

Serão apresentados o valor mensal total da carteira, a receita mensal associada aos clientes ativos, a receita mensal potencialmente perdida e o percentual que essa perda representa em relação ao total.

Esses indicadores permitirão avaliar a dimensão financeira dos cancelamentos e apoiar as próximas análises.

In [4]:

receita_ativa = tabela_receita["No"]
receita_perdida = tabela_receita["Yes"]

receita_total = receita_ativa + receita_perdida

percentual_perda = (receita_perdida / receita_total * 100).round(2)

print(f"Valor mensal total da carteira: R$ {receita_total:.2f}")
print(f"Receita mensal dos clientes ativos: R$ {receita_ativa:.2f}")
print(f"Receita mensal potencialmente perdida: R$ {receita_perdida:.2f}")
print(f"Percentual de receita potencialmente perdida: {percentual_perda:.2f}%")


Valor mensal total da carteira: R$ 364010.05
Receita mensal dos clientes ativos: R$ 251767.15
Receita mensal potencialmente perdida: R$ 112242.90
Percentual de receita potencialmente perdida: 30.84%


### Interpretação dos indicadores financeiros

O valor mensal total associado à carteira analisada é de R$ 364.010,05, sendo R$ 251.767,15 referentes aos clientes que permaneceram e R$ 112.242,90 aos clientes que cancelaram os serviços.

Os valores mensais associados aos clientes que cancelaram representam aproximadamente 30,84% do total da carteira analisada.

Esse resultado demonstra a relevância financeira dos cancelamentos e indica a importância de identificar quais grupos de clientes concentram os maiores valores de receita potencialmente perdida.

Vale destacar que os valores representam uma estimativa baseada nas mensalidades dos clientes, não sendo possível determinar as perdas financeiras efetivamente registradas em um período específico.